# NSE Buyback PDF Downloader (Colab)
Downloads **3,741** NSE Buyback filings (PDFs + ZIPs→PDFs) from NSE Archives → Google Drive.

**Drive path:** `My Drive → ArthLLM-2B → nse_buyback pdfs`

**Run Cell 0 → 1 → 2 → 3 → 4**

In [1]:
# CELL 0: Keepalive — prevents Colab from disconnecting
import threading, time
def _ka():
    while True: time.sleep(55); _ = 1+1
if not any(t.name=='keepalive' for t in threading.enumerate()):
    threading.Thread(target=_ka, name='keepalive', daemon=True).start()
    print('✅ Keepalive started')
else:
    print('✅ Keepalive already running')

✅ Keepalive started


In [2]:
# CELL 1: Mount Google Drive (smart — skips if already mounted)
import os, shutil, time
from google.colab import drive
MOUNT = '/content/drive'
already_ok = False
try:
    tp = os.path.join(MOUNT, 'MyDrive')
    if os.path.isdir(tp) and os.listdir(tp): already_ok = True
except: pass
if already_ok:
    print('✅ Drive already mounted — skipping.')
else:
    if os.path.isdir(MOUNT):
        try: shutil.rmtree(MOUNT, ignore_errors=True)
        except: pass
    for attempt in range(1, 6):
        try:
            drive.mount(MOUNT)
            print('✅ Drive mounted!')
            break
        except Exception as e:
            if 'already contain' in str(e): shutil.rmtree(MOUNT, ignore_errors=True)
            if attempt < 5: print(f'Mount {attempt}/5 failed: {e}, retry in {10*attempt}s'); time.sleep(10*attempt)
            else: raise

Mounted at /content/drive
✅ Drive mounted!


In [4]:
# CELL 2: Locate nse_buyback.csv and set up output directory
import os

# ── Direct path (no recursive walk) ──────────────────────────────────────────
BASE         = '/content/drive/MyDrive/ArthLLM-2B/nse_buyback pdfs'
MANIFEST_CSV = os.path.join(BASE, 'nse_buyback.csv')
OUT_DIR      = os.path.join(BASE, 'raw_pdfs')

if not os.path.isfile(MANIFEST_CSV):
    raise FileNotFoundError(
        f'nse_buyback.csv not found at expected path:\n  {MANIFEST_CSV}\n'
        'Please upload nse_buyback.csv to: MyDrive > ArthLLM-2B > nse_buyback pdfs'
    )

os.makedirs(OUT_DIR, exist_ok=True)

print(f'✅ Manifest : {MANIFEST_CSV}')
print(f'📥 Output   : {OUT_DIR}')

✅ Manifest : /content/drive/MyDrive/ArthLLM-2B/nse_buyback pdfs/nse_buyback.csv
📥 Output   : /content/drive/MyDrive/ArthLLM-2B/nse_buyback pdfs/raw_pdfs


In [5]:
# CELL 3: Fast Resume — two-way sync between CSV status and files on disk
import os, re
import pandas as pd
from tqdm.notebook import tqdm

# ── helpers ──────────────────────────────────────────────────────────────────
def safe(s, mx=80):
    """Sanitize string for use in file/folder names."""
    return re.sub(r'[<>:"/\\|?*\s]+', '_', str(s).strip())[:mx]

def expected_filenames(row):
    """Return the set of filenames this row would produce on disk."""
    url = str(row.get('pdf_url', '')).strip()
    if not url or url == '-': return set()
    basename = os.path.splitext(os.path.basename(url))[0]  # strip .pdf / .zip
    sym  = safe(row.get('symbol', 'UNK'))
    seq  = str(row.get('seq_id', '')).split('.')[0]         # drop .0
    stem = f'{sym}_{seq}_{basename}'[:150]
    if url.endswith('.pdf'):
        return {stem + '.pdf'}
    else:  # .zip — may contain multiple PDFs; we track the zip sentinel
        return {stem + '.zip_done'}   # sentinel file written after zip extract

# ── load CSV ─────────────────────────────────────────────────────────────────
fsize = os.path.getsize(MANIFEST_CSV) if os.path.exists(MANIFEST_CSV) else 0
if fsize == 0: raise RuntimeError('CORRUPT CSV: 0 bytes')
try:
    df = pd.read_csv(MANIFEST_CSV, dtype={'seq_id': str})
except Exception as e:
    raise RuntimeError(f'CORRUPT CSV: {e}')
if len(df) == 0: raise RuntimeError('CSV has 0 rows')
print(f'Loaded {len(df)} rows')

# ── init / normalise status column ───────────────────────────────────────────
DONE_VALS = ('true', 'done', 'skip')
if 'dl_status' not in df.columns:
    df['dl_status'] = 'pending'
else:
    df['dl_status'] = df['dl_status'].apply(
        lambda v: 'done' if str(v).strip().lower() in DONE_VALS
        else (str(v).strip().lower() if pd.notna(v) and str(v).strip() else 'pending')
    )

# rows with no valid URL → mark skip immediately
no_url_mask = df['pdf_url'].isna() | (df['pdf_url'].astype(str).str.strip() == '-')
df.loc[no_url_mask & (df['dl_status'] == 'pending'), 'dl_status'] = 'no_url'

# ── scan existing files on Drive ─────────────────────────────────────────────
print('Scanning for existing files on Drive…')
existing = set()
if os.path.exists(OUT_DIR):
    for r, d, f in os.walk(OUT_DIR):
        for x in f: existing.add(x)
print(f'Found {len(existing)} files on Drive')

# ── two-way sync ──────────────────────────────────────────────────────────────
updated = 0
for i in tqdm(range(len(df)), desc='Syncing manifest'):
    row  = df.iloc[i]
    fnames = expected_filenames(row)
    cur = str(df.at[df.index[i], 'dl_status']).lower()
    file_exists = bool(fnames & existing)
    if cur == 'done' and not file_exists:
        df.at[df.index[i], 'dl_status'] = 'pending'; updated += 1
    elif cur not in ('done', 'no_url') and file_exists:
        df.at[df.index[i], 'dl_status'] = 'done'; updated += 1

if updated > 0:
    df.to_csv(MANIFEST_CSV, index=False, quoting=1)
    print(f'Updated {updated} rows in manifest')
else:
    print('Manifest already in sync')

counts = df['dl_status'].value_counts()
print('\nStatus breakdown:')
for s, c in counts.items(): print(f'  {s}: {c}')

Loaded 4154 rows
Scanning for existing files on Drive…
Found 0 files on Drive


Syncing manifest:   0%|          | 0/4154 [00:00<?, ?it/s]

Manifest already in sync

Status breakdown:
  pending: 3741
  no_url: 413


In [6]:
# CELL 4: Download NSE Buyback PDFs / ZIPs
# ─ PDFs  → saved directly as SYMBOL_SEQID_origname.pdf
# ─ ZIPs  → extracted; all PDFs inside saved; sentinel .zip_done written
# ─ Resume-safe: skips files already on disk; autosaves manifest every 300 s
# ─ 15 parallel workers, 4 retries, exponential backoff on 429/503

import pandas as pd, requests, os, time, re, threading, zipfile, io
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm

# ── config ───────────────────────────────────────────────────────────────────
WORKERS    = 15
SAVE_EVERY = 100        # save manifest every N completions
TIMER_SAVE = 300        # autosave every 5 minutes
TIMEOUT    = 45
MAX_RETRIES= 4
HEADERS    = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/124.0.0.0 Safari/537.36',
    'Accept': 'application/pdf,application/zip,*/*',
    'Referer': 'https://www.nseindia.com/'
}

# ── helpers ───────────────────────────────────────────────────────────────────
def safe(s, mx=80):
    return re.sub(r'[<>:"/\\|?*\s]+', '_', str(s).strip())[:mx]

def is_valid_pdf(data):
    if not data or len(data) < 64: return False
    if data[:4] == b'%PDF': return True
    if b'<html' in data[:512].lower() or b'<!doctype' in data[:512].lower(): return False
    return len(data) > 4096

def is_valid_zip(data):
    return data and len(data) > 22 and data[:2] == b'PK'

def download_one(url, symbol, seq_id, folder):
    """
    Download url into folder.
    Returns: 'ok', 'skip', 'fail', or 'http_NNN'
    """
    os.makedirs(folder, exist_ok=True)
    basename  = os.path.splitext(os.path.basename(url))[0]
    sym_safe  = safe(symbol)
    seq_safe  = str(seq_id).split('.')[0]
    stem      = f'{sym_safe}_{seq_safe}_{basename}'[:150]
    is_zip    = url.lower().endswith('.zip')
    sentinel  = os.path.join(folder, stem + '.zip_done')
    pdf_dest  = os.path.join(folder, stem + '.pdf')

    # ── skip if already done ──────────────────────────────────────────────────
    if is_zip and os.path.exists(sentinel):  return 'skip'
    if not is_zip and os.path.exists(pdf_dest) and os.path.getsize(pdf_dest) > 1024:
        return 'skip'

    # ── fetch with retries ────────────────────────────────────────────────────
    data = None
    for attempt in range(MAX_RETRIES):
        try:
            r = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
            if r.status_code == 200:
                data = r.content; break
            if r.status_code in (429, 503):
                time.sleep(5 * (2 ** attempt)); continue
            return f'http_{r.status_code}'
        except requests.RequestException:
            time.sleep(2 * (attempt + 1))
    if data is None: return 'fail'

    # ── handle PDF ────────────────────────────────────────────────────────────
    if not is_zip:
        if not is_valid_pdf(data): return 'invalid'
        with open(pdf_dest, 'wb') as f: f.write(data)
        return 'ok'

    # ── handle ZIP → extract PDFs ─────────────────────────────────────────────
    if not is_valid_zip(data): return 'invalid_zip'
    try:
        with zipfile.ZipFile(io.BytesIO(data)) as zf:
            pdf_members = [n for n in zf.namelist() if n.lower().endswith('.pdf')]
            if not pdf_members:
                # no PDFs — save all files anyway
                pdf_members = zf.namelist()
            for member in pdf_members:
                safe_member = safe(os.path.basename(member), mx=120)
                out_path = os.path.join(folder, f'{stem}_{safe_member}')
                with zf.open(member) as src, open(out_path, 'wb') as dst:
                    dst.write(src.read())
        # write sentinel so we never re-extract
        with open(sentinel, 'w') as f: f.write('done')
        return 'ok'
    except zipfile.BadZipFile:
        return 'bad_zip'

# ── load CSV ──────────────────────────────────────────────────────────────────
fsize = os.path.getsize(MANIFEST_CSV) if os.path.exists(MANIFEST_CSV) else 0
if fsize == 0: raise RuntimeError('CORRUPT CSV: 0 bytes')
try:
    df = pd.read_csv(MANIFEST_CSV, dtype={'seq_id': str})
except Exception as e:
    raise RuntimeError(f'CORRUPT CSV: {e}')

DONE_VALS = ('true', 'done', 'skip')
if 'dl_status' not in df.columns:
    df['dl_status'] = 'pending'
else:
    df['dl_status'] = df['dl_status'].apply(
        lambda v: 'done' if str(v).strip().lower() in DONE_VALS
        else (str(v).strip().lower() if pd.notna(v) and str(v).strip() else 'pending')
    )

# ── build task list (pending rows with valid URLs only) ───────────────────────
pending = df[
    (df['dl_status'] == 'pending') &
    df['pdf_url'].notna() &
    (df['pdf_url'].astype(str).str.strip() != '-')
]
print(f'Pending: {len(pending)} / {len(df)}')

if len(pending) == 0:
    print('🎉 All done!')
else:
    tasks = []
    for idx, row in pending.iterrows():
        url    = str(row['pdf_url']).strip()
        sym    = str(row.get('symbol', 'UNK')).strip()
        seq    = str(row.get('seq_id', 'noseq')).split('.')[0]
        co     = safe(str(row.get('company', sym)), mx=60)
        folder = os.path.join(OUT_DIR, f'{safe(sym)}_{co}')
        tasks.append((idx, url, sym, seq, folder))

    ok = skip = fail = counter = 0
    _stop = threading.Event()
    _lock = threading.Lock()

    def _autosave():
        while not _stop.wait(TIMER_SAVE):
            with _lock:
                try: df.to_csv(MANIFEST_CSV, index=False, quoting=1)
                except: pass
    threading.Thread(target=_autosave, daemon=True, name='autosave').start()

    try:
        with ThreadPoolExecutor(max_workers=WORKERS) as pool:
            futs = {
                pool.submit(download_one, u, sym, seq, fld): (i, u)
                for i, u, sym, seq, fld in tasks
            }
            pbar = tqdm(total=len(futs), desc='NSE Buyback PDFs', unit='file')
            for fut in as_completed(futs):
                i, u = futs[fut]
                try:   res = fut.result()
                except: res = 'fail'

                if res in ('ok', 'skip'):
                    ok   += 1 if res == 'ok' else 0
                    skip += 1 if res == 'skip' else 0
                    with _lock: df.at[i, 'dl_status'] = 'done'
                else:
                    fail += 1
                    with _lock: df.at[i, 'dl_status'] = res   # e.g. 'http_404'

                counter += 1
                if counter % SAVE_EVERY == 0:
                    with _lock: df.to_csv(MANIFEST_CSV, index=False, quoting=1)

                pbar.set_postfix(ok=ok, skip=skip, fail=fail)
                pbar.update(1)
            pbar.close()

    except KeyboardInterrupt:
        print('⚠️  Ctrl+C — saving manifest…')
    finally:
        _stop.set()
        with _lock: df.to_csv(MANIFEST_CSV, index=False, quoting=1)
        print(f'💾 Manifest saved: {MANIFEST_CSV}')

    total  = len(df)
    done_n = (df['dl_status'] == 'done').sum()
    print(f'\n✅ SUMMARY  Total={total}  Done={done_n}  OK={ok}  Skip={skip}  Fail={fail}  Pending={total - done_n}')

Pending: 3741 / 4154


NSE Buyback PDFs:   0%|          | 0/3741 [00:00<?, ?file/s]

💾 Manifest saved: /content/drive/MyDrive/ArthLLM-2B/nse_buyback pdfs/nse_buyback.csv

✅ SUMMARY  Total=4154  Done=3727  OK=3727  Skip=0  Fail=14  Pending=427
